# Aula 6 — Visualização com ggplot2
**Introdução à Programação para Pesquisa Biomédica · IBCCF/UFRJ**

> Runtime → Change runtime type → R

In [ ]:
library(tidyverse)
library(patchwork)

set.seed(42)
n <- 80
df <- tibble(
  gene      = paste0("GENE", sprintf("%03d",1:n)),
  organismo = sample(c("Homo sapiens","Mus musculus"), n, replace=TRUE),
  condicao  = sample(c("controle","tratamento"), n, replace=TRUE),
  expressao = round(rlnorm(n, 1.5, 0.8), 3)
) |> mutate(log2_expr=log2(expressao),
            gc_pct=round(runif(n,35,70),1))
glimpse(df)

## 1. Scatter plot

In [ ]:
ggplot(df, aes(x=gc_pct, y=log2_expr,
               color=organismo, shape=condicao)) +
  geom_point(alpha=0.75, size=2.5) +
  labs(title="GC% vs. Expressão (log2)",
       x="Conteúdo GC (%)",
       y="Expressão (log2)") +
  theme_classic(base_size=13)

## 2. Histograma + densidade

In [ ]:
ggplot(df, aes(x=log2_expr, fill=condicao)) +
  geom_histogram(bins=20, alpha=0.7,
                 position="identity") +
  geom_density(aes(y=after_stat(count)),
               alpha=0.3) +
  labs(title="Distribuição de Expressão",
       x="Expressão (log2)") +
  theme_classic()

## 3. Boxplot + pontos

In [ ]:
ggplot(df, aes(x=organismo, y=log2_expr,
               fill=condicao)) +
  geom_boxplot(outlier.shape=NA, alpha=0.7,
               position=position_dodge(0.8)) +
  geom_jitter(aes(color=condicao),
              position=position_jitterdodge(0.2),
              alpha=0.4, size=1.5) +
  labs(title="Expressão por Organismo",
       x=NULL, y="Expressão (log2)") +
  theme_classic()

## 4. Heatmap de correlação

In [ ]:
library(reshape2)

df_wide <- df |>
  mutate(row_id=row_number()) |>
  pivot_wider(id_cols=row_id,
              names_from=gene,
              values_from=log2_expr) |>
  select(-row_id) |>
  cor(use="pairwise.complete.obs")

df_melt <- melt(df_wide[1:8,1:8])

ggplot(df_melt, aes(Var1, Var2, fill=value)) +
  geom_tile() +
  geom_text(aes(label=round(value,2)), size=2.5) +
  scale_fill_gradient2(low="blue",mid="white",
                       high="red", midpoint=0) +
  labs(title="Correlação entre genes") +
  theme_minimal() +
  theme(axis.text.x=element_text(angle=45,hjust=1))

## 5. Figura combinada e exportação

In [ ]:
p1 <- ggplot(df, aes(x=condicao, y=log2_expr,
                    fill=condicao)) +
  geom_boxplot(outlier.shape=NA, alpha=0.8) +
  geom_jitter(width=0.2, alpha=0.4, size=1.5) +
  labs(title="Expressão por Condição",
       x=NULL, y="Expressão (log2)") +
  theme_classic() + theme(legend.position="none")

p2 <- ggplot(df, aes(x=gc_pct, y=log2_expr,
                     color=organismo)) +
  geom_point(alpha=0.7, size=2) +
  labs(title="GC% vs. Expressão") +
  theme_classic()

p_final <- p1 + p2 +
  plot_annotation(title="Análise de Expressão Gênica")

ggsave("/tmp/figura_final.png", p_final,
       width=12, height=5, dpi=300)
print(p_final)